# App Review Scraper
Scrapes reviews from App Store and Play Store across multiple markets.

**Author:** Hugo Malorey  
**Stack:** requests, google-play-scraper, pandas  
**Output schema:** `date | review | rating | source | language | country | answered | username`

> Note: App Store title is concatenated into `review` field for homogeneity.

## 1. Imports

In [23]:
import pandas as pd
import requests
from datetime import datetime, timedelta, timezone
from google_play_scraper import reviews, Sort

## 2. Config

### Appstore Scraping via iTunes RSS feed

The [App Store Connect API](https://developer.apple.com/documentation/appstoreconnectapi/get-v1-apps-_id_-customerreviews) is the official Apple API but requires JWT authentication and access to the target app's App Store Connect account — so it only works if you own the app.

The iTunes RSS feed (`itunes.apple.com/{country}/rss/customerreviews/...`) is an undocumented but long-standing public endpoint that works for **any app** with no auth. Since this scraper is meant to be app-agnostic, the RSS feed is the practical choice. Caveat: it's capped at ~500 reviews per country (10 pages × 50 reviews).

### Playstore Reviews Scraping

For the Play Store, we use the [`google-play-scraper`](https://github.com/JoMingyu/google-play-scraper) Python library, which wraps the same unofficial endpoint the Play Store web UI uses — also no auth required.

## 3. Test connections
Verify both stores are reachable before full scrape.

In [24]:
# ── Time window ──────────────────────────────────────────────
DAYS_BACK   = 90
CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)

# ── App identifiers ──────────────────────────────────────────
APP_NAME      = "airbnb"
APP_STORE_ID  = "401626263"
PLAY_STORE_ID = "com.airbnb.android"

# ── Market presets ───────────────────────────────────────────
MARKETS_FR = [
    {"country": "fr", "language": "fr"},
]

MARKETS_EU_MAIN = [
    {"country": "fr", "language": "fr"},  # France
    {"country": "de", "language": "de"},  # Germany
    {"country": "es", "language": "es"},  # Spain
    {"country": "it", "language": "it"},  # Italy
    {"country": "nl", "language": "nl"},  # Netherlands
    {"country": "be", "language": "fr"},  # Belgium
    {"country": "pt", "language": "pt"},  # Portugal
    {"country": "at", "language": "de"},  # Austria
]

MARKETS_EU_ALL = [
    {"country": "fr", "language": "fr"},  # France
    {"country": "de", "language": "de"},  # Germany
    {"country": "es", "language": "es"},  # Spain
    {"country": "it", "language": "it"},  # Italy
    {"country": "nl", "language": "nl"},  # Netherlands
    {"country": "be", "language": "fr"},  # Belgium
    {"country": "pt", "language": "pt"},  # Portugal
    {"country": "at", "language": "de"},  # Austria
    {"country": "pl", "language": "pl"},  # Poland
    {"country": "se", "language": "sv"},  # Sweden
    {"country": "dk", "language": "da"},  # Denmark
    {"country": "fi", "language": "fi"},  # Finland
    {"country": "ie", "language": "en"},  # Ireland
    {"country": "gr", "language": "el"},  # Greece
    {"country": "ro", "language": "ro"},  # Romania
    {"country": "hu", "language": "hu"},  # Hungary
    {"country": "cz", "language": "cs"},  # Czech Republic
    {"country": "sk", "language": "sk"},  # Slovakia
    {"country": "hr", "language": "hr"},  # Croatia
    {"country": "bg", "language": "bg"},  # Bulgaria
    {"country": "lt", "language": "lt"},  # Lithuania
    {"country": "lv", "language": "lv"},  # Latvia
    {"country": "ee", "language": "et"},  # Estonia
    {"country": "si", "language": "sl"},  # Slovenia
    {"country": "lu", "language": "fr"},  # Luxembourg
    {"country": "mt", "language": "en"},  # Malta
    {"country": "cy", "language": "el"},  # Cyprus
]

# ── Active market selection ───────────────────────────────────
# Switch between MARKETS_FR / MARKETS_EU_MAIN / MARKETS_EU_ALL
MARKETS = MARKETS_FR

# ── Rating filters ───────────────────────────────────────────
MIN_RATING = 1
MAX_RATING = 5

# ── Output columns ───────────────────────────────────────────
COLUMNS = ["date", "review", "rating", "source", "language", "country", "answered", "username"]

print(f"App:            {APP_NAME}")
print(f"Period:         Last {DAYS_BACK} days (since {CUTOFF_DATE.strftime('%Y-%m-%d')})")
print(f"Rating filter:  {MIN_RATING} to {MAX_RATING} stars")
print(f"Markets:        {[m['country'] for m in MARKETS]}")

App:            airbnb
Period:         Last 90 days (since 2026-01-10)
Rating filter:  1 to 5 stars
Markets:        ['fr']


In [25]:
test_market = MARKETS[0]  # use the first market in the list as a quick sanity check

# ── Test App Store ────────────────────────────────────────────
print("=== App Store connection test ===")
url = (
    f"https://itunes.apple.com/{test_market['country']}/rss/customerreviews"
    f"/page=1/id={APP_STORE_ID}/sortby=mostrecent/json"
)  # Apple's public RSS feed endpoint — returns JSON with the first page of reviews for this app/country
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)

# User-Agent header avoids getting blocked by Apple's servers; timeout prevents hanging indefinitely
entries  = response.json().get("feed", {}).get("entry", [])

# "feed.entry" is where Apple nests the review objects in the RSS JSON response
print(f"Status:         {response.status_code}")
print(f"Reviews found:  {len(entries)} on page 1")

if entries:
    s       = entries[0]  # grab the most recent review
    title   = s.get("title", {}).get("label", "")    # App Store reviews have a separate title field
    content = s.get("content", {}).get("label", "")  # the body text of the review
    review  = " — ".join(filter(None, [title, content]))  # concat title + body, skipping blanks
    print(f"Latest date:    {s.get('updated', {}).get('label', 'N/A')[:10]}")   # slice to YYYY-MM-DD
    print(f"Latest rating:  {s.get('im:rating', {}).get('label', 'N/A')}★")    # rating lives under "im:rating"
    print(f"Sample review:  {review[:100]}...")  # truncate to keep output readable

print()

# ── Test Play Store ───────────────────────────────────────────
print("=== Play Store connection test ===")
result, _ = reviews(
    PLAY_STORE_ID,
    lang=test_market["language"],
    country=test_market["country"],
    sort=Sort.NEWEST,  # sort by newest so we get the most recent review
    count=1            # only fetch 1 review — just enough to confirm the connection works
)
# the second return value is a continuation token for pagination; we don't need it here
print(f"Status:         OK")
print(f"Reviews found:  {len(result)} fetched")
if result:
    s = result[0]
    print(f"Latest date:    {s['at'].strftime('%Y-%m-%d')}")  # 'at' is the review's datetime object
    print(f"Latest rating:  {s['score']}★")                  # 'score' is the star rating (1–5)
    print(f"Sample review:  {str(s['content'])[:100]}...")    # 'content' is the review body text

=== App Store connection test ===
Status:         200
Reviews found:  50 on page 1
Latest date:    2026-04-08
Latest rating:  1★
Sample review:  Bug — Il y a énormément de bug sur l’appli en ce moment, au bout d’une minute l’appli se ferme autom...

=== Play Store connection test ===
Status:         OK
Reviews found:  1 fetched
Latest date:    2026-04-09
Latest rating:  3★
Sample review:  compliqué pour accéder à une réservation de villa...


## 4. Scrape App Store (iOS)

In [26]:
def scrape_app_store(app_store_id, markets, cutoff_date, min_rating, max_rating, pages=5):
    """Scrape App Store reviews. Title concatenated into review field."""
    results = []
    for market in markets:
        country, language = market["country"], market["language"]
        print(f"  [{country}] scraping...")
        count = 0
        for page in range(1, pages + 1):
            url = (
                f"https://itunes.apple.com/{country}/rss/customerreviews"
                f"/page={page}/id={app_store_id}/sortby=mostrecent/json"
            )
            try:
                resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
            except requests.exceptions.Timeout:
                print(f"  [{country}] page {page} timeout — stopping")
                break

            if resp.status_code != 200:
                break

            entries = resp.json().get("feed", {}).get("entry", [])
            if not entries:
                break

            # Apple returns a dict instead of list when only 1 entry — normalize
            if isinstance(entries, dict):
                entries = [entries]

            stop = False
            for entry in entries:
                rating   = int(entry.get("im:rating", {}).get("label", 5))
                date_str = entry.get("updated", {}).get("label", "")[:10]
                date     = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)

                if date < cutoff_date:
                    stop = True
                    break

                if not (min_rating <= rating <= max_rating):
                    continue

                title   = entry.get("title", {}).get("label", "")
                content = entry.get("content", {}).get("label", "")
                review  = " — ".join(filter(None, [title, content]))

                results.append({
                    "date":     date,
                    "review":   review,
                    "rating":   rating,
                    "source":   "App Store",
                    "language": language,
                    "country":  country,
                    "answered": False,
                    "username": entry.get("author", {}).get("name", {}).get("label", "anonymous"),
                })
                count += 1

            if stop:
                print(f"  [{country}] cutoff reached at page {page} — stopping")
                break

        print(f"  [{country}] → {count} reviews")
    return results


print("Scraping App Store...")
ios_reviews = scrape_app_store(APP_STORE_ID, MARKETS, CUTOFF_DATE, MIN_RATING, MAX_RATING)
print(f"Total iOS: {len(ios_reviews)}")

Scraping App Store...
  [fr] scraping...
  [fr] cutoff reached at page 4 — stopping
  [fr] → 150 reviews
Total iOS: 150


## 5. Scrape Play Store (Android)

In [27]:
def scrape_play_store(play_store_id, markets, cutoff_date, min_rating, max_rating, count=300):
    """Scrape Play Store reviews. No title field on this platform."""
    results = []
    for market in markets:
        country, language = market["country"], market["language"]
        print(f"  [{country}] scraping...")
        market_count = 0
        for star in range(min_rating, max_rating + 1):
            fetched, _ = reviews(
                play_store_id,
                lang=language,
                country=country,
                sort=Sort.NEWEST,
                count=count,
                filter_score_with=star,
            )
            star_count = 0
            for r in fetched:
                date = r["at"]
                if date.tzinfo is None:
                    date = date.replace(tzinfo=timezone.utc)
                if date < cutoff_date:
                    continue
                results.append({
                    "date":     date,
                    "review":   r.get("content", ""),
                    "rating":   r["score"],
                    "source":   "Play Store",
                    "language": language,
                    "country":  country,
                    "answered": r.get("replyContent") is not None,
                    "username": r.get("userName", "anonymous"),
                })
                star_count   += 1
                market_count += 1
            print(f"  [{country}] {star}★: {star_count} reviews")
        print(f"  [{country}] → {market_count} total")
    return results


print("Scraping Play Store...")
android_reviews = scrape_play_store(PLAY_STORE_ID, MARKETS, CUTOFF_DATE, MIN_RATING, MAX_RATING)
print(f"Total Android: {len(android_reviews)}")

Scraping Play Store...
  [fr] scraping...
  [fr] 1★: 95 reviews
  [fr] 2★: 19 reviews
  [fr] 3★: 24 reviews
  [fr] 4★: 110 reviews
  [fr] 5★: 300 reviews
  [fr] → 548 total
Total Android: 548


## 6. Merge & clean

In [28]:
df = pd.DataFrame(ios_reviews + android_reviews, columns=COLUMNS)
df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("date", ascending=False).reset_index(drop=True)

date_min = df["date"].min().strftime("%Y-%m-%d")
date_max = df["date"].max().strftime("%Y-%m-%d")

rating_counts = df["rating"].value_counts().sort_index()
rating_pcts   = (rating_counts / len(df) * 100).round(1)

print(f"{'='*40}")
print(f"{APP_NAME.upper()} — {date_min} to {date_max}")
print(f"{'='*40}")
print(f"TOTAL:           {len(df)} reviews")
print(f"Avg rating:      {df['rating'].mean():.2f}★")
print(f"App Store:       {len(df[df['source'] == 'App Store'])}")
print(f"Play Store:      {len(df[df['source'] == 'Play Store'])}")
print(f"Answered:        {df['answered'].sum()} ({df['answered'].mean()*100:.0f}%)")
print()
print("Rating distribution:")
for star, count in rating_counts.items():
    print(f"  {star}★  {count:>4}  ({rating_pcts[star]}%)")
print()
df.head(20)


AIRBNB — 2026-01-10 to 2026-04-09
TOTAL:           698 reviews
Avg rating:      3.38★
App Store:       150
Play Store:      548
Answered:        2 (0%)

Rating distribution:
  1★   221  (31.7%)
  2★    27  (3.9%)
  3★    28  (4.0%)
  4★   112  (16.0%)
  5★   310  (44.4%)



,date,review,rating,source,language,country,answered,username
0,2026-04-09 11:57:48+00:00,compliqué pour accéder à une réservation de villa,3,Play Store,fr,fr,False,Reine Vitry
1,2026-04-09 09:04:42+00:00,Des gens bizarres chez qui nous avions pris un...,1,Play Store,fr,fr,False,Patricia Meheut
2,2026-04-09 08:47:47+00:00,réservation simple de fonctionnement,5,Play Store,fr,fr,False,Séréna de Barros
3,2026-04-08 21:42:12+00:00,excellent,5,Play Store,fr,fr,False,Yassir Sami
4,2026-04-08 21:25:37+00:00,bonne expérience,5,Play Store,fr,fr,False,Fabienne Estivalet-Elias
5,2026-04-08 20:59:50+00:00,Pas mal,3,Play Store,fr,fr,False,Caroline Boverie
6,2026-04-08 20:44:05+00:00,un choix pléthorique. Des critères précis. Séc...,5,Play Store,fr,fr,False,Magaly Mavilia
7,2026-04-08 19:09:25+00:00,parfait comme d'habitude.,4,Play Store,fr,fr,False,Raphaël Raph (“Raph”)
8,2026-04-08 18:31:24+00:00,super,5,Play Store,fr,fr,False,Danny Bonneau
9,2026-04-08 17:53:23+00:00,Super expérience !,5,Play Store,fr,fr,False,Aurélie


## 6b. All-time ratings vs last 90 days

In [29]:
from google_play_scraper import app as play_app

ios_rows  = []
play_rows = []

for market in MARKETS:
    country, language = market["country"], market["language"]

    # ── App Store ─────────────────────────────────────────────
    try:
        lookup    = requests.get(
            f"https://itunes.apple.com/lookup?id={APP_STORE_ID}&country={country}",
            timeout=5
        ).json().get("results", [{}])[0]
        ios_alltime = round(lookup.get("averageUserRating", 0), 2)
        ios_count   = lookup.get("userRatingCount", 0)
    except Exception:
        ios_alltime, ios_count = None, None

    df_ios   = df[(df["country"] == country) & (df["source"] == "App Store")]
    ios_90d  = round(df_ios["rating"].mean(), 2) if len(df_ios) else None
    ios_rows.append({"country": country, "alltime": ios_alltime, "alltime_n": ios_count, f"{DAYS_BACK}d_avg": ios_90d, f"{DAYS_BACK}d_n": len(df_ios)})

    # ── Play Store ────────────────────────────────────────────
    try:
        play_data   = play_app(PLAY_STORE_ID, lang=language, country=country)
        play_alltime = round(play_data.get("score") or 0, 2)
        play_count   = play_data.get("ratings", 0)
    except Exception:
        play_alltime, play_count = None, None

    df_play   = df[(df["country"] == country) & (df["source"] == "Play Store")]
    play_90d  = round(df_play["rating"].mean(), 2) if len(df_play) else None
    play_rows.append({"country": country, "alltime": play_alltime, "alltime_n": play_count, f"{DAYS_BACK}d_avg": play_90d, f"{DAYS_BACK}d_n": len(df_play)})

ios_summary  = pd.DataFrame(ios_rows)
play_summary = pd.DataFrame(play_rows)

print(f"App Store — {APP_NAME.upper()}")
print(ios_summary.to_string(index=False))
print()
print(f"Play Store — {APP_NAME.upper()}")
print(play_summary.to_string(index=False))

App Store — AIRBNB
country  alltime  alltime_n  90d_avg  90d_n
     fr     4.65     138002     1.41    150

Play Store — AIRBNB
country  alltime  alltime_n  90d_avg  90d_n
     fr     4.47    1883052     3.91    548


## 6c. AI Analysis — Top issues from 1★ & 2★ reviews

In [32]:
import anthropic
from dotenv import load_dotenv

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

# ── Filter 1★ and 2★ reviews ─────────────────────────────────
df_negative = df[df["rating"] <= 2].copy()
n_negative  = len(df_negative)

# Format reviews for Claude
review_lines = []
for _, row in df_negative.iterrows():
    review_lines.append(
        f"[{row['source']} | {row['rating']}★ | {row['country'].upper()} | {row['date'].strftime('%Y-%m-%d')}]\n"
        f"{row['review']}"
    )
reviews_text = "\n\n".join(review_lines)

print(f"Sending {n_negative} reviews (1★ & 2★) to Claude for analysis...")
print(f"Total characters: {len(reviews_text)}\n")

# ── Call Claude API ───────────────────────────────────────────
client = anthropic.Anthropic()

prompt = f"""Below are {n_negative} negative reviews (1 and 2 stars) for the app "{APP_NAME}" over the last {DAYS_BACK} days.

Identify the TOP 5 most frequently mentioned issues. For each issue:
- Give it a clear, short name
- Describe what users are complaining about (1-2 sentences)
- Count how many of the {n_negative} reviews mention this issue (approximate is fine)

Format your response exactly like this for each issue:
## 1. [Issue Name]
**Reviews mentioning this: X/{n_negative} (~Y%)**
[Description]

---

Here are the reviews:

{reviews_text}"""

with client.messages.stream(
    model="claude-opus-4-6",
    max_tokens=4096,
    messages=[{"role": "user", "content": prompt}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

Sending 248 reviews (1★ & 2★) to Claude for analysis...
Total characters: 67167

## 1. iOS 18 / Compatibility with Older Devices
**Reviews mentioning this: ~75/248 (~30%)**
Users are unable to use the Airbnb app after a recent update that requires iOS 18, rendering the app inaccessible on older iPhones (iPhone X, iPhone 8, iPhone 7, etc.) and iPads. Many users feel forced into buying a new device and are switching to competitors like Booking.com.

---

## 2. Poor Customer Service / Support
**Reviews mentioning this: ~55/248 (~22%)**
Users report that Airbnb's customer support is unhelpful, unresponsive, and incompetent — particularly when dealing with disputes, refund requests, or safety issues. Agents give generic responses, make promises they don't keep, and fail to resolve problems.

---

## 3. Unfair Dispute Resolution / Lack of Protection (Hosts & Guests)
**Reviews mentioning this: ~45/248 (~18%)**
Both hosts and guests complain that Airbnb sides against them in disputes — guests 

## 7. Print all reviews (readable)

In [15]:
for i, row in df.iterrows():
    print(f"{'─'*60}")
    print(f"#{i+1} [{row['source']} | {row['rating']}★ | {row['country'].upper()} | {row['date'].strftime('%Y-%m-%d')} | answered: {row['answered']}]")
    print(f"Review:  {row['review']}")
    print(f"User:    {row['username']}")
print(f"{'─'*60}")
print(f"TOTAL: {len(df)} reviews")

────────────────────────────────────────────────────────────
#1 [Play Store | 3★ | FR | 2026-04-09 | answered: False]
Review:  compliqué pour accéder à une réservation de villa
User:    Reine Vitry
────────────────────────────────────────────────────────────
#2 [Play Store | 1★ | FR | 2026-04-09 | answered: False]
Review:  Des gens bizarres chez qui nous avions pris une location la dernière fois. On a desinstallé l'application du coup.
User:    Patricia Meheut
────────────────────────────────────────────────────────────
#3 [Play Store | 5★ | FR | 2026-04-09 | answered: False]
Review:  réservation simple de fonctionnement
User:    Séréna de Barros
────────────────────────────────────────────────────────────
#4 [Play Store | 5★ | FR | 2026-04-08 | answered: False]
Review:  excellent
User:    Yassir Sami
────────────────────────────────────────────────────────────
#5 [Play Store | 5★ | FR | 2026-04-08 | answered: False]
Review:  bonne expérience
User:    Fabienne Estivalet-Elias
─────────

## 8. Export to CSV

In [16]:
output_file = f"{APP_NAME}_reviews_{datetime.now().strftime('%Y%m%d')}_{DAYS_BACK}d.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved → {output_file}")

Saved → airbnb_reviews_20260410_90d.csv


## 9. Format for Claude analysis

In [17]:
lines = []
for _, row in df.iterrows():
    answered_part = " [answered]" if row['answered'] else ""
    lines.append(
        f"[{row['source']} | {row['rating']}★ | {row['country'].upper()} | "
        f"{row['date'].strftime('%Y-%m-%d')}{answered_part}]\n"
        f"{row['review']}\n"
    )

claude_input = "\n".join(lines)

print(f"Total reviews:    {len(df)}")
print(f"Total characters: {len(claude_input)}")
print()
print("--- SAMPLE (first 3) ---")
print("\n".join(lines[:3]))

Total reviews:    698
Total characters: 112719

--- SAMPLE (first 3) ---
[Play Store | 3★ | FR | 2026-04-09]
compliqué pour accéder à une réservation de villa

[Play Store | 1★ | FR | 2026-04-09]
Des gens bizarres chez qui nous avions pris une location la dernière fois. On a desinstallé l'application du coup.

[Play Store | 5★ | FR | 2026-04-09]
réservation simple de fonctionnement

